# import the needed liberaries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from datetime import date
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
from sklearn.model_selection import train_test_split

# get the data set

In [2]:
df = pd.read_csv('healthcare_dataset.csv')

In [3]:
df.head()

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,1/31/2024,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2/2/2024,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,8/20/2019,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,8/26/2019,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,9/22/2022,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,10/7/2022,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,11/18/2020,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,12/18/2020,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,9/19/2022,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,10/9/2022,Penicillin,Abnormal


In [4]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [5]:
print(X)

[['Bobby JacksOn' 30 'Male' ... 'Urgent' '2/2/2024' 'Paracetamol']
 ['LesLie TErRy' 62 'Male' ... 'Emergency' '8/26/2019' 'Ibuprofen']
 ['DaNnY sMitH' 76 'Female' ... 'Emergency' '10/7/2022' 'Aspirin']
 ...
 ['HEATher WaNG' 38 'Female' ... 'Urgent' '8/10/2020' 'Ibuprofen']
 ['JENniFER JOneS' 43 'Male' ... 'Elective' '5/31/2019' 'Ibuprofen']
 ['jAMES GARCiA' 53 'Female' ... 'Urgent' '4/29/2024' 'Ibuprofen']]


In [6]:
print(y)

['Normal' 'Inconclusive' 'Normal' ... 'Abnormal' 'Abnormal' 'Abnormal']


## Identify missing values and deal with them

In [7]:
df.isnull().sum()

Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
dtype: int64

In [8]:
# we do not have missing values in the data set

# Remove unneeded variables

In [8]:
#we will remove the variables we don't need like name and hospital
df = df.drop(columns=['Name', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number','Date of Admission','Discharge Date'])

In [9]:
df.head()

,Age,Gender,Blood Type,Medical Condition,Admission Type,Medication,Test Results
0,30,Male,B-,Cancer,Urgent,Paracetamol,Normal
1,62,Male,A+,Obesity,Emergency,Ibuprofen,Inconclusive
2,76,Female,A-,Obesity,Emergency,Aspirin,Normal
3,28,Female,O+,Diabetes,Elective,Ibuprofen,Abnormal
4,43,Female,AB+,Cancer,Urgent,Penicillin,Abnormal


# Encode Catagorical Data

In [10]:
#we will use get dummies for the encoding
pd.get_dummies(df)

,Age,Gender_Female,Gender_Male,Blood Type_A+,Blood Type_A-,Blood Type_AB+,Blood Type_AB-,Blood Type_B+,Blood Type_B-,Blood Type_O+,...,Admission Type_Emergency,Admission Type_Urgent,Medication_Aspirin,Medication_Ibuprofen,Medication_Lipitor,Medication_Paracetamol,Medication_Penicillin,Test Results_Abnormal,Test Results_Inconclusive,Test Results_Normal
0,30,False,True,False,False,False,False,False,True,False,...,False,True,False,False,False,True,False,False,False,True
1,62,False,True,True,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,True,False
2,76,True,False,False,True,False,False,False,False,False,...,True,False,True,False,False,False,False,False,False,True
3,28,True,False,False,False,False,False,False,False,True,...,False,False,False,True,False,False,False,True,False,False
4,43,True,False,False,False,True,False,False,False,False,...,False,True,False,False,False,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55495,42,True,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,True,True,False,False
55496,61,True,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,True
55497,38,True,False,False,False,False,False,True,False,False,...,False,True,False,True,False,False,False,True,False,False
55498,43,False,True,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False


In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Age                55500 non-null  int64 
 1   Gender             55500 non-null  object
 2   Blood Type         55500 non-null  object
 3   Medical Condition  55500 non-null  object
 4   Admission Type     55500 non-null  object
 5   Medication         55500 non-null  object
 6   Test Results       55500 non-null  object
dtypes: int64(1), object(6)
memory usage: 3.0+ MB
None


# Now we need to split the Data Set

In [12]:
#we will split the data set into 80% Training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 48)

In [ ]:
print(X_train)

[['MAUReen SmiTh' 42 'Female' ... 'Emergency' '10/29/2020' 'Ibuprofen']
 ['peTER WHite' 35 'Male' ... 'Emergency' '5/1/2021' 'Ibuprofen']
 ['RYAN bEnnETt' 42 'Male' ... 'Elective' '8/15/2021' 'Lipitor']
 ...
 ['sHAron fOstER' 37 'Male' ... 'Elective' '8/22/2022' 'Lipitor']
 ['AnTHONY CArrILLO' 83 'Female' ... 'Urgent' '6/23/2021' 'Penicillin']
 ['DaNIEL baker' 71 'Male' ... 'Emergency' '9/7/2021' 'Aspirin']]


In [14]:
print(X_test)

[['MEGHan mURrAY' 55 'Male' ... 'Elective' '7/13/2023' 'Penicillin']
 ['wilLIAm smIth' 28 'Female' ... 'Urgent' '2/9/2020' 'Penicillin']
 ['kRisTiN ArIAS' 64 'Female' ... 'Emergency' '8/8/2020' 'Penicillin']
 ...
 ['STePhAniE STePheNsOn' 30 'Female' ... 'Elective' '4/2/2022' 'Aspirin']
 ['STevEN MAsoN' 33 'Female' ... 'Urgent' '1/28/2024' 'Paracetamol']
 ['tYRone RuSSelL' 50 'Female' ... 'Urgent' '4/2/2024' 'Ibuprofen']]
